In [33]:
import numpy as np
import pandas as pd
import lseg.data as ld
import matplotlib.pyplot as plt
import time
from pathlib import Path

In [34]:
# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [35]:
# open Refinitiv session
ld.open_session()

<lseg.data.session.Definition object at 0x2d84dfbeea0 {name='workspace'}>

In [36]:
FRQ = {"daily": "D", "weekly": "W", "monthly": "M"}

JOBS = {
    "price":  {"field": "TR.PriceClose",                     "interval": "monthly", "batch_size": 50},
    "volume": {"field": "TR.Volume",                         "interval": "daily",   "batch_size": 10},
    "marketcap": {"field": "TR.CompanyMarketCap",       "interval": "monthly", "batch_size": 50},
}

In [37]:
def to_long(df):
    shaped_df = df.stack(level=0)
    shaped_df = shaped_df.reset_index()
    shaped_df = shaped_df.rename(columns={"level_1": "ric", "Date": "date"})
    return shaped_df

In [38]:
# Create the directory name
def raw_dir(index_ric, start, end, label):
    name = f"{index_ric.lstrip('.')}_{start}_{end}_{label}"
    return Path("data/raw") / name

In [39]:
# Show stocks listed on an exchange each month (or whatever frequency is chosen)
def get_members(index_ric, start, end, freq="ME"):
    # 1. All joiners and leavers between start and end
    const_changes = ld.get_data(
        universe=[index_ric],
        fields=["TR.IndexJLConstituentChangeDate", "TR.IndexJLConstituentRIC", 
                "TR.IndexJLConstituentName", "TR.IndexJLConstituentituentChange"],
        parameters={"SDATE":f"{start}","EDATE":f"{end}", 'IC':'B'}
    )
    const_changes["Date"] = pd.to_datetime(const_changes["Date"])

    # 2. Members ON the start date: the members field, not the changes field
    start_members = ld.get_data(
        universe=[index_ric], 
        fields=["TR.IndexConstituentRIC"], 
        parameters={"SDATE":f"{start}"}
    )
    init_list = start_members["Constituent RIC"].dropna().tolist()

    # 3. Replay history month by month
    rows = []
    prev = pd.to_datetime(start)
    for m in pd.date_range(start, end, freq=freq):
        in_window = (const_changes["Date"] > prev) & (const_changes["Date"] <= m)
        window = const_changes[in_window]
        window = window.sort_values("Date")

        for ric, change in zip(window["Constituent RIC"], window["Change"]):
            if change == "Leaver" and ric in init_list:
                init_list.remove(ric)
            elif change == "Joiner" and ric not in init_list:
                init_list.append(ric)

        for ric in init_list:
            rows.append({"Date": m, "ric": ric, "in_index": True})
        prev = m

    return pd.DataFrame(rows)

def members_path(index_ric, start, end):
    folder = raw_dir(index_ric, start, end, "members")
    folder.mkdir(parents=True, exist_ok=True)
    return folder / "members.parquet"


def save_members(members, index_ric, start, end):
    path = members_path(index_ric, start, end)
    members.rename(columns={"Date": "date"}).to_parquet(path)
    print(f"saved {path} ({len(members)} rows)")


def load_or_get_members(index_ric, start, end):
    path = members_path(index_ric, start, end)
    if path.exists():
        return pd.read_parquet(path)
    members = get_members(index_ric, start, end)
    save_members(members, index_ric, start, end)
    return members.rename(columns={"Date": "date"})

In [40]:
def fetch_with_retry(batch, field, interval, start, end, tries=3):
    for attempt in range(tries):
        try:
            return ld.get_data(
                universe=batch,
                fields=[f"{field}.date", field],
                parameters={"SDate": start, "EDate": end, "Frq": FRQ[interval]}
            )
        except Exception as e:
            print(f"attempt {attempt + 1} failed: {e}. Waiting {2 ** attempt}s")
            time.sleep(2 ** attempt)
    return None
        

def download_all(index_ric, rics, label, start, end, wait=0.3):
    job = JOBS[label]
    size = job["batch_size"]
    batches = [rics[i: i+size] for i in range(0, len(rics), size)]
    failed = []

    out_dir = raw_dir(index_ric, start, end, label)
    out_dir.mkdir(parents=True, exist_ok=True)

    for index, b in enumerate(batches):
        # Create the parquet filename
        path = out_dir / f"batch_{index:04d}.parquet"
        if path.exists(): 
            continue

        df = fetch_with_retry(b, job["field"], job["interval"], start, end)
        if df is None or df.empty:
            failed.append((index, b))
            continue

        df.columns = ["ric", "date", label]
        df["date"] = pd.to_datetime(df["date"])

        # Create a tmp file in case it gets interrupted
        tmp = path.with_suffix(".tmp")
        df.to_parquet(tmp)
        tmp.rename(path)
        print(f"saved {path.name} ({len(df)} rows)")
        
        # Pause to reduce requests to Refinitiv per second
        time.sleep(wait)

    return failed

In [41]:
def load_all(index_ric, start, end, label):
    folder = raw_dir(index_ric, start, end, label)
    tables = [pd.read_parquet(p) for p in sorted(folder.glob("*.parquet"))]
    if not tables:
        raise ValueError(f"no files in {folder}")
    panel = pd.concat(tables, ignore_index=True)
    return panel.sort_values(["ric", "date"]).reset_index(drop=True)

In [42]:
def data_report(df, value_col):
    counts = df.groupby("ric").size()

    real = df.dropna(subset=[value_col])
    real_counts = real.groupby("ric").size()

    return {
        "value_col": value_col,
        "rows": len(df),
        "distinct rics": df["ric"].nunique(),
        "missing values": df[value_col].isna().sum(),
        "first date": df["date"].min(),
        "last date": df["date"].max(),
        "missing dates": df["date"].isna().sum(),
        "missing date and value": (df["date"].isna() & df[value_col].isna()).sum(),
        "duplicates": df[["date", "ric"]].duplicated().sum(),
        "duplicates (dated rows only)": df.dropna(subset=["date"]).duplicated(["date", "ric"]).sum(),
        "zero values": (df[value_col] == 0).sum(),
        "minimum value": df[value_col].min(),
        "maximum value": df[value_col].max(),
        "median value": df[value_col].median(),
        "ric of minimum": df.loc[df[value_col].idxmin(), "ric"],
        "ric of maximum": df.loc[df[value_col].idxmax(), "ric"],
        "min real rows per ric": real_counts.min(),
        "maximum real rows per ric": real_counts.max(),
        "minimum distinct counts": counts.min(),
        "maximum distinct counts": counts.max(),
        "median distinct counts": counts.median(),
        "weekend dates": (df["date"].dt.dayofweek >= 5).sum()
    }

def print_report(report):
    print(f"--- {report['value_col']} ---")
    for key, value in report.items():
        print(f"{key:>28}: {value}")

In [43]:
def drop_undated(df):
    return df[df["date"].notna()]

def drop_repeats(df):
    return df[~df.duplicated(["ric", "date"])]

def to_month_end(df):
    df = df.copy()
    df["date"] = df["date"] + pd.offsets.MonthEnd(0)
    return df

def zero_to_nan(df, col):
    df = df.copy()
    df[col] = df[col].replace(0, np.nan)
    return df

def monthly_volume(df):
    df = to_month_end(df)
    return df.groupby(["date", "ric"])["volume"].sum(min_count=1).reset_index()

In [44]:
index_ric = ".NDX"
start = "2020-01-01"
end = "2026-01-01"

# Load the members in the exchange per month
members = load_or_get_members(index_ric, start, end)

# Get all members that has ever been in the exchange
rics = sorted(members["ric"].unique().tolist())

# Download data from Refinitiv
for label in JOBS:
    failed = download_all(index_ric, rics, label, start, end)
    print(label, "failed:", failed)

# Load the data from the files
price = load_all(index_ric, start, end, "price")
volume = load_all(index_ric, start, end, "volume")
marketcap = load_all(index_ric, start, end, "marketcap")

saved data\raw\NDX_2020-01-01_2026-01-01_members\members.parquet (7313 rows)
price failed: []
volume failed: []
marketcap failed: []


In [45]:
# Generate data report to look for anomalies
price_data_report = data_report(price, "price")
volume_data_report = data_report(volume, "volume")
marketcap_data_report = data_report(marketcap, "marketcap")

print_report(price_data_report)
print_report(volume_data_report)
print_report(marketcap_data_report)

--- price ---
                   value_col: price
                        rows: 11088
               distinct rics: 154
              missing values: 426
                  first date: 2020-01-31 00:00:00
                   last date: 2025-12-31 00:00:00
               missing dates: 373
      missing date and value: 373
                  duplicates: 672
duplicates (dated rows only): 312
                 zero values: 0
               minimum value: 3.11
               maximum value: 2613.63
                median value: 105.64500000000001
              ric of minimum: PTON.OQ
              ric of maximum: MELI.OQ
       min real rows per ric: 3
   maximum real rows per ric: 72
     minimum distinct counts: 72
     maximum distinct counts: 72
      median distinct counts: 72.0
               weekend dates: 0
--- volume ---
                   value_col: volume
                        rows: 232232
               distinct rics: 154
              missing values: 0
                  first dat

In [ ]:
# Clean the data
def clean_data(price, volume, marketcap):
    price_cleaned = to_month_end(drop_repeats(drop_undated(price)))
    volume_cleaned = monthly_volume(zero_to_nan(drop_repeats(drop_undated(volume)), "volume"))
    marketcap_cleaned = to_month_end(drop_repeats(drop_undated(marketcap)))
    return (price_cleaned, volume_cleaned, marketcap_cleaned)


def create_panel(price_cleaned, volume_cleaned, marketcap_cleaned):
    panel = price_cleaned.merge(volume_cleaned, on=["ric", "date"], how="left")
    panel = panel.merge(marketcap_cleaned, on=["ric", "date"], how="left")
    panel = panel.merge(to_month_end(members), on=["ric", "date"], how="left")
    panel = panel.sort_values(["ric", "date"])
    panel["in_index"] = panel["in_index"].fillna(False).astype(bool)
    panel["ret"] = panel.groupby("ric")["price"].pct_change()
    panel["turnover"] = panel["price"] * panel["volume"] / panel["marketcap"]
    return panel


def save_panel(panel, name="panel"):
    out_dir = Path("data") / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{name}.parquet"
    tmp = path.with_suffix(".tmp")
    panel.to_parquet(tmp, index=False)
    tmp.replace(path)
    return path


price_cleaned, volume_cleaned, marketcap_cleaned = clean_data(price, volume, marketcap)
panel = create_panel(price_cleaned, volume_cleaned, marketcap_cleaned)
save_panel(panel)

WindowsPath('data/processed/panel.parquet')